# Adrianic v4.4 — Multi-Consequence Value + Adaptive Binding Gate

This notebook integrates:

- multi-consequence consequence-earned memory value,
- latent dependency/common-cause discounting,
- separate novelty and interference pressures,
- adaptive comparative replacement,
- parity-protected raw templates,
- honest unresolved associative addressing,
- first-only / second-only / reverse / matched-wrong binding controls,
- a dual-lane anti-scalarization ablation.

No semantic/fact/chain labels enter memory valuation or replacement.

In [ ]:
from __future__ import annotations
from pathlib import Path
from collections import Counter, defaultdict
import argparse, copy, json, math, random, re, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# -----------------------------------------------------------------------------
# Adrianic v4.4 — Multi-consequence value + adaptive-gate binding test:
#   (1) pairwise + latent common-cause evidence control (v3.21 family)
#   (2) comparative replacement + honest address/template separation (field gate)
#
# Scientific boundary: this is a tiny recurrent LM + explicit memory controller.
# It tests functional relational binding under interference, not human understanding.
# -----------------------------------------------------------------------------

if Path('/content').exists(): ROOT = Path('/content')
elif Path('/mnt/data').exists(): ROOT = Path('/mnt/data')
else: ROOT = Path.cwd()
OUT = ROOT / 'adrianic_v44_multiconsequence_results'
OUT.mkdir(parents=True, exist_ok=True)

TOKEN_RE = re.compile(r"[a-zA-Z]+|[0-9]+|[^\w\s]")
STOP = set('the a an is are in of to and this that with for near under before after through beyond fact remember'.split())

def seed_all(seed:int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def tokenize(s:str): return TOKEN_RE.findall(s.lower())

def content_tokens(s:str):
    return [w for w in tokenize(s) if w.isalpha() and w not in STOP]

FACTS=[
('two plus two equals','four'),('three plus five equals','eight'),('seven minus three equals','four'),('six times seven equals','fortytwo'),
('the square root of nine is','three'),('a triangle has','three'),('a square has','four'),('the additive identity is','zero'),
('the si unit of force is the','newton'),('the si unit of energy is the','joule'),('the si unit of power is the','watt'),('mass times acceleration gives','force'),
('energy associated with motion is','kinetic'),('the force attracting masses is','gravity'),('the chemical symbol for hydrogen is','h'),
('the chemical symbol for oxygen is','o'),('water contains hydrogen and','oxygen'),('an atom with six protons is','carbon'),
('an atom with eight protons is','oxygen'),('the center of an atom is the','nucleus'),('earth orbits the','sun'),('the moon orbits the','earth'),
('the largest planet in the solar system is','jupiter'),('plants use light in','photosynthesis')]
REPHRASE=[
('in mechanics mass times acceleration produces','force'),('force is measured using the unit called','newton'),
('energy uses the si unit called','joule'),('water contains hydrogen plus','oxygen'),('six protons identifies','carbon'),
('eight protons identifies','oxygen'),('earth travels around the','sun'),('the moon travels around the','earth')]

SYL1=['nar','vel','tor','lum','kas','mer','sol','dav','pir','zan','fel','rin','bos','kel','vim','jor','tav','qul','ser','pan','dru','hel','cor','nis']
SYL2=['a','en','or','is','um','et','on','ar','il','os','ex','un']

def unique_words(n,seed):
    rng=random.Random(seed); pool=[a+b for a in SYL1 for b in SYL2]; rng.shuffle(pool); return pool[:n]

def make_chains(seed=0,n_demo=20,n_probe=10):
    names=unique_words((n_demo+n_probe)*3,seed+818); demo=[]; probe=[]; rows=[]; k=0
    for i in range(n_demo+n_probe):
        x,y,z=names[k:k+3]; k+=3
        premises=[f'{x} contains {y}.',f'{y} maps to {z}.']
        if i<n_demo:
            # Preserve v4.2 curriculum exactly: sentences are independent training items.
            demo += (premises+[f'{x} ultimately maps to {z}.',f'{y} is contained by {x}.'])*3
            rows.append(('demo',x,y,z))
        else:
            # Probe target and inverse sentences are deliberately absent.
            probe += premises*4
            rows.append(('probe',x,y,z))
    return demo,probe,pd.DataFrame(rows,columns=['kind','x','y','z'])

def make_training_corpus(seed=0,contextual_demo=False):
    rng=random.Random(seed); s=[]
    for p,a in FACTS: s += [f'{p} {a}.',f'fact: {p} {a}.',f'remember {p} {a}.']
    demo,probe,chains=make_chains(seed); s += demo+probe
    if contextual_demo:
        # Schema-ingestion patch: demonstration entities additionally appear in a contiguous
        # premise->premise->composition context. Probe entities still NEVER receive the
        # composed target or a combined two-premise paragraph during training.
        for _,r in chains[chains.kind=='demo'].iterrows():
            para=f'{r.x} contains {r.y}. {r.y} maps to {r.z}. {r.x} ultimately maps to {r.z}.'
            s += [para,para]
    for i in range(80): s.append(f'noise{i} drift{i} hollow{i} amber{i} sevenfold{i}.')
    rng.shuffle(s)
    return s,chains

def proxy_stream(seed=0,n=180):
    rng=random.Random(seed+500)
    names=['arden','cassian','lyra','orion','mara','selene','tavian','viola']
    verbs=['whispers','wanders','argues','laughs','waits','returns','dreams','speaks']
    objs=['beside the gate','through the hall','near the river','under the tower','after the feast','beyond the garden','before the crowd','under the moon']
    return [f'{rng.choice(names)} {rng.choice(verbs)} {rng.choice(objs)}.' for _ in range(n)]

def download_natural():
    urls={'shakespeare':'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
          'bible':'https://www.gutenberg.org/cache/epub/10900/pg10900.txt'}
    out={}
    for name,url in urls.items():
        p=ROOT/f'{name}_corpus.txt'
        if not p.exists():
            print('downloading',name); urllib.request.urlretrieve(url,p)
        text=p.read_text(errors='ignore')
        out[name]=[q.strip().replace('\n',' ') for q in re.split(r'(?<=[.!?])\s+|\n{2,}',text) if 4<=len(tokenize(q))<=70]
    return out

def natural_stream(seed=0,corpus='interleaved',per_source=700):
    d=download_natural(); rng=random.Random(seed+900)
    A=rng.sample(d['shakespeare'],min(per_source,len(d['shakespeare'])))
    B=rng.sample(d['bible'],min(per_source,len(d['bible'])))
    if corpus=='shakespeare': return A
    if corpus=='bible': return B
    out=[]; n=min(len(A),len(B))
    for a,b in zip(A[:n],B[:n]): out += ([a,b] if rng.random()<.5 else [b,a])
    return out

class Vocab:
    def __init__(self,texts,max_vocab=10000):
        c=Counter()
        for t in texts:c.update(tokenize(t))
        self.itos=['<pad>','<unk>']+[w for w,_ in c.most_common(max_vocab-2)]
        self.stoi={w:i for i,w in enumerate(self.itos)}
    def encode(self,s):return [self.stoi.get(w,1) for w in tokenize(s)]

class TinySSM(nn.Module):
    def __init__(self,V,emb=36,hid=56):
        super().__init__();self.emb=nn.Embedding(V,emb);self.rnn=nn.RNN(emb,hid,nonlinearity='tanh',batch_first=True);self.out=nn.Linear(hid,V)
    def forward(self,x,h=None):z,h=self.rnn(self.emb(x),h);return self.out(z),h

def dev(model):return next(model.parameters()).device

def train_sentence(model,opt,ids):
    if len(ids)<2:return np.nan
    d=dev(model);x=torch.tensor(ids[:-1],dtype=torch.long,device=d)[None,:];y=torch.tensor(ids[1:],dtype=torch.long,device=d)[None,:]
    opt.zero_grad(set_to_none=True);logits,_=model(x);loss=nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1));loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step();return float(loss.item())

def seq_loss(model,ids):
    if len(ids)<2:return 0.
    d=dev(model);x=torch.tensor(ids[:-1],dtype=torch.long,device=d)[None,:];y=torch.tensor(ids[1:],dtype=torch.long,device=d)[None,:]
    with torch.no_grad():logits,_=model(x)
    return float(nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1)).item())

def continuation_loss(model,vocab,prefix,target):
    """Cross-entropy on target tokens only, after conditioning on prefix."""
    pids=vocab.encode(prefix);tids=vocab.encode(target)
    if not pids or not tids:return 0.0
    ids=pids+tids;d=dev(model)
    x=torch.tensor(ids[:-1],dtype=torch.long,device=d)[None,:]
    y=torch.tensor(ids[1:],dtype=torch.long,device=d)[None,:]
    with torch.no_grad():logits,_=model(x)
    start=max(0,len(pids)-1);end=min(logits.shape[1],start+len(tids))
    if end<=start:return 0.0
    return float(nn.functional.cross_entropy(logits[:,start:end,:].reshape(-1,logits.size(-1)),y[:,start:end].reshape(-1)).item())

def logits_for_prompt(model,vocab,prompt):
    ids=vocab.encode(prompt);d=dev(model);x=torch.tensor(ids,dtype=torch.long,device=d)[None,:]
    with torch.no_grad():logits,_=model(x)
    return logits[0,-1]

def token_rank(model,vocab,prompt,answer):
    v=logits_for_prompt(model,vocab,prompt);aid=vocab.stoi.get(answer,1);return int((v>v[aid]).sum().item())+1

def top_tokens(model,vocab,prompt,k=5):
    v=logits_for_prompt(model,vocab,prompt);return [vocab.itos[i] for i in torch.topk(v,k=min(k,len(vocab.itos))).indices.tolist()]

def candidate_rank(model,vocab,prompt,true_answer,candidates):
    v=logits_for_prompt(model,vocab,prompt)
    scores=[]
    for a in candidates:
        scores.append((a,float(v[vocab.stoi.get(a,1)].item())))
    scores.sort(key=lambda x:x[1],reverse=True)
    order=[a for a,_ in scores]
    return order.index(true_answer)+1,dict(scores)

# -----------------------------------------------------------------------------
# Candidate evidence and v3.21-style dependency/factor inference
# -----------------------------------------------------------------------------

def unique_candidate_table(train,vocab,early_model,mid_model,final_model):
    """Build candidate evidence from observed consequences, without semantic labels."""
    pos=defaultdict(list)
    for i,s in enumerate(train):pos[s].append(i)
    docs=list(pos.keys());N=len(train)
    df=Counter()
    for s in docs:df.update(set(content_tokens(s)))
    token_sets=[set(content_tokens(s)) for s in train]
    rows=[]
    for idx,s in enumerate(docs):
        toks=content_tokens(s);st=set(toks);pp=pos[s];first=min(pp);last=max(pp)
        recurrence=math.log1p(len(pp))
        token_persistence=float(np.mean([math.log1p(df[t]) for t in toks])) if toks else 0.
        cross_context=float(np.mean([df[t]>=3 for t in toks])) if toks else 0.
        temporal_span=(last-first)/max(1,N-1) if len(pp)>1 else 0.
        early_pred=-seq_loss(early_model,vocab.encode(s));mid_pred=-seq_loss(mid_model,vocab.encode(s));final_pred=-seq_loss(final_model,vocab.encode(s))
        gain_early_mid=mid_pred-early_pred;gain_mid_final=final_pred-mid_pred

        future_idx=list(range(first+1,N));supports=[]
        if st and future_idx:
            for w in np.array_split(np.asarray(future_idx,dtype=int),3):
                seen=set()
                for j in w.tolist():seen |= token_sets[j]
                supports.append(len(st&seen)/max(1,len(st)))
        future_support=float(np.mean(supports)) if supports else 0.

        maxjac=0.;seen_prev=set()
        if st:
            for j in range(first):
                if train[j] in seen_prev:continue
                seen_prev.add(train[j]);ps=token_sets[j]
                if ps:maxjac=max(maxjac,len(st&ps)/(len(st|ps)+1e-12))
        novelty=1.-maxjac

        rel=[];unrel=[]
        for j in range(first+1,N):
            if train[j]==s:continue
            ov=len(st&token_sets[j]) if st else 0
            if ov>0:rel.append((ov/(len(st|token_sets[j])+1e-12),j))
            elif len(unrel)<6:unrel.append(j)
        rg=[]
        for _,j in sorted(rel,reverse=True)[:2]:
            t=train[j];rg.append(seq_loss(final_model,vocab.encode(t))-continuation_loss(final_model,vocab,s,t))
        context_gain=float(np.mean(rg)) if rg else 0.
        ug=[]
        for j in unrel[:2]:
            t=train[j];ug.append(continuation_loss(final_model,vocab,s,t)-seq_loss(final_model,vocab.encode(t)))
        low_interference=-float(np.mean(ug)) if ug else 0.

        rows.append({'uid':idx,'text':s,'first_pos':first,'last_pos':last,'count':len(pp),
                     'recurrence':recurrence,'token_persistence':token_persistence,'cross_context':cross_context,
                     'temporal_span':temporal_span,'early_predictability':early_pred,'mid_predictability':mid_pred,'future_predictability':final_pred,
                     'gain_early_mid':gain_early_mid,'gain_mid_final':gain_mid_final,'future_support':future_support,
                     'context_gain':context_gain,'novelty':novelty,'low_interference':low_interference})
    return pd.DataFrame(rows)

SOURCE_COLS=['recurrence','token_persistence','cross_context','temporal_span','early_predictability']
CONSEQUENCE_COLS=['gain_early_mid','gain_mid_final','future_support','context_gain']

def zcols(X):
    X=np.asarray(X,float);mu=X.mean(0);sd=X.std(0);active=sd>1e-8;Z=np.zeros_like(X)
    Z[:,active]=(X[:,active]-mu[active])/(sd[active]+1e-9)
    return Z,mu,sd,active

def connected_components_from_corr(C,threshold=.72):
    n=C.shape[0];adj={i:set() for i in range(n)}
    for i in range(n):
        for j in range(i+1,n):
            if C[i,j]>threshold:adj[i].add(j);adj[j].add(i)
    comps=[];seen=set()
    for i in range(n):
        if i in seen:continue
        st=[i];c=set()
        while st:
            x=st.pop()
            if x in seen:continue
            seen.add(x);c.add(x);st.extend(adj[x]-seen)
        if len(c)>=2:comps.append(c)
    return comps

def learn_multifactor(cov,maxf=3):
    n=cov.shape[0];vals,vecs=np.linalg.eigh(cov);o=np.argsort(vals)[::-1];vals=vals[o];vecs=vecs[:,o]
    base=np.median(vals[min(maxf,n):])+1e-9 if n>maxf else 1e-9
    groups=[];loads=[]
    for k in range(min(maxf,n)):
        strength=max(0.,float((vals[k]-base)/(vals[k]+1e-9)))
        v=np.abs(vecs[:,k]);mx=v.max()+1e-12;g={i for i,l in enumerate(v) if l/mx>.58}
        if len(g)>=2 and strength>.22 and not any(len(g&h)/len(g|h)>.75 for h in groups):
            groups.append(g);loads.append((v,strength))
    return groups,loads,vals

def cap_weights(raw,g):
    g=[i for i in g if raw[i]>0]
    if len(g)<=1:return raw
    total=sum(raw[i] for i in g);cap=max(raw[i] for i in g);f=cap/(total+1e-12)
    for i in g:raw[i]*=f
    return raw

def evidence_model(cand,mode='factor'):
    X=cand[SOURCE_COLS].to_numpy(float);Z,mu,sd,active=zcols(X)
    y=cand['future_predictability'].to_numpy(float);yz=(y-y.mean())/(y.std()+1e-9)
    residual=Z-yz[:,None]
    bias=residual.mean(0)
    rho=np.mean(np.exp(-np.abs(residual)/.8),axis=0)
    cov=np.cov((residual-residual.mean(0)).T)+np.eye(len(SOURCE_COLS))*1e-6
    rsd=np.sqrt(np.maximum(np.diag(cov),1e-9));corr=np.clip(cov/(rsd[:,None]*rsd[None,:]+1e-12),-1,1)
    comps=connected_components_from_corr(corr,.72)
    groups,loads,eigs=learn_multifactor(cov,3)
    raw={i:float(rho[i]**3) if active[i] else 0. for i in range(len(SOURCE_COLS))}
    if mode in ('graph','factor'):
        for g in comps:raw=cap_weights(raw,g)
    if mode=='factor':
        for g,(load,strength) in zip(groups,loads):
            raw=cap_weights(raw,g);mx=max(load[i] for i in g)+1e-12
            for i in g:raw[i]/=(1+1.15*strength*load[i]/mx)
    w=np.array([raw[i] for i in range(len(SOURCE_COLS))]);w/=w.sum()+1e-12
    deb=Z-bias[None,:];U=deb@w
    disagreement=np.sum(w[None,:]*(deb-U[:,None])**2,axis=1)
    # v2-style common-mode risk: don't 'correct' value; lower confidence when many channels miss together.
    substantial=np.abs(residual)>.8
    same_sign=np.maximum((residual>.8).sum(1),(residual<-.8).sum(1))
    h=float(np.mean(same_sign>=max(2,int(math.ceil(len(SOURCE_COLS)/2)))))
    conf=np.exp(-disagreement)*(1-min(.8,h))
    U01=1/(1+np.exp(-np.clip(U,-12,12)))
    out=cand.copy();out['utility_z']=U;out['utility01']=U01;out['evidence_confidence']=conf;out['future_yz']=yz
    meta={'source_cols':SOURCE_COLS,'rho':rho.tolist(),'bias':bias.tolist(),'weights':w.tolist(),'corr':corr.tolist(),
          'graph_components':[[SOURCE_COLS[i] for i in sorted(g)] for g in comps],
          'factor_groups':[[SOURCE_COLS[i] for i in sorted(g)] for g in groups],
          'factor_strengths':[float(x[1]) for x in loads],'factor_loadings':[x[0].tolist() for x in loads],
          'common_cause_risk':h,'eigenvalues':eigs.tolist()}
    return out,meta

def rank_centered(x):
    x=np.asarray(x,float);r=pd.Series(x).rank(method='average',pct=True).to_numpy(float)
    return 2.*(r-.5)

def multi_consequence_model(cand,mode='factor'):
    """Dependency-discount multiple directly observed consequence channels."""
    Y=np.column_stack([rank_centered(cand[c].to_numpy(float)) for c in CONSEQUENCE_COLS])
    active=np.std(Y,axis=0)>1e-8
    cov=np.cov(Y.T)+np.eye(len(CONSEQUENCE_COLS))*1e-6
    sd=np.sqrt(np.maximum(np.diag(cov),1e-9));corr=np.clip(cov/(sd[:,None]*sd[None,:]+1e-12),-1,1)
    comps=connected_components_from_corr(corr,.72);groups,loads,eigs=learn_multifactor(cov,3)
    raw={i:(1. if active[i] else 0.) for i in range(len(CONSEQUENCE_COLS))}
    if mode in ('graph','factor'):
        for g in comps:raw=cap_weights(raw,g)
    if mode=='factor':
        for g,(load,strength) in zip(groups,loads):
            raw=cap_weights(raw,g);mx=max(load[i] for i in g)+1e-12
            for i in g:raw[i]/=(1+1.15*strength*load[i]/mx)
    w=np.array([raw[i] for i in range(len(CONSEQUENCE_COLS))],float);w/=w.sum()+1e-12
    U=Y@w;disagreement=np.sum(w[None,:]*(Y-U[:,None])**2,axis=1)
    same_bad=(Y<-.40).sum(1);h=float(np.mean(same_bad>=max(2,int(math.ceil(len(CONSEQUENCE_COLS)/2)))))
    conf=np.exp(-disagreement)*(1-min(.8,h));U01=1/(1+np.exp(-2*np.clip(U,-6,6)))
    out=cand.copy();out['utility_z']=U;out['utility01']=U01;out['evidence_confidence']=conf
    # Keep novelty and interference structurally distinct from consequence value.
    # Novelty can soften a replacement wall, but it cannot make a bad candidate good.
    # Interference is a cost/constraint, not a positive evidence vote.
    out['novelty01']=(rank_centered(out['novelty'].to_numpy(float))+1.)/2.
    out['interference_risk01']=(rank_centered(-out['low_interference'].to_numpy(float))+1.)/2.
    meta={'consequence_cols':CONSEQUENCE_COLS,'weights':w.tolist(),'corr':corr.tolist(),
          'graph_components':[[CONSEQUENCE_COLS[i] for i in sorted(g)] for g in comps],
          'factor_groups':[[CONSEQUENCE_COLS[i] for i in sorted(g)] for g in groups],
          'factor_strengths':[float(x[1]) for x in loads],'factor_loadings':[x[0].tolist() for x in loads],
          'common_cause_risk':h,'eigenvalues':eigs.tolist()}
    return out,meta

def portfolio_model(cand_v43,cand_multi):
    """Keep alternative consequence models as parallel support lanes.

    A candidate may earn permanence from either lane; one scalar model is not
    allowed to erase a structurally different useful profile. No semantic labels
    define the lanes: one is the prior predictive structural estimator, the other
    is direct multi-consequence evidence.
    """
    a=cand_v43.sort_values('uid').reset_index(drop=True);b=cand_multi.sort_values('uid').reset_index(drop=True)
    assert np.array_equal(a.uid.to_numpy(),b.uid.to_numpy())
    out=b.copy();ua=a.utility01.to_numpy(float);ub=b.utility01.to_numpy(float);qa=a.evidence_confidence.to_numpy(float);qb=b.evidence_confidence.to_numpy(float)
    choose_b=ub>=ua
    out['utility_v43']=ua;out['utility_multi']=ub;out['confidence_v43']=qa;out['confidence_multi']=qb
    out['utility01']=np.maximum(ua,ub);out['evidence_confidence']=np.where(choose_b,qb,qa);out['portfolio_lane']=np.where(choose_b,'multi','v43')
    return out

# -----------------------------------------------------------------------------
# Parity-protected raw templates + comparative replacement bank
# Address != template: address is transparent IDF token signature; template is raw ids.
# -----------------------------------------------------------------------------

class Slot:
    PRIME=1000003
    def __init__(self,text,ids,uid,U,S,Q,count,first_pos,created_step):
        self.text=text;self.a=list(ids);self.b=list(ids);self.uid=uid;self.U=float(U);self.S=float(S);self.Q=float(Q);self.count=int(count);self.first_pos=int(first_pos);self.created_step=int(created_step)
        self.p,self.q=self.chk(self.a);self.corrupted=False
    @classmethod
    def chk(cls,ids):
        q=np.asarray(ids,dtype=np.int64);return int(q.sum()%cls.PRIME),int(((np.arange(len(q))+1)*q).sum()%cls.PRIME)
    def corrupt(self,rng):
        if self.a:
            j=rng.randrange(len(self.a));self.a[j]=(self.a[j]+rng.randint(1,19))%100000;self.corrupted=True
    def retrieve(self,parity=True):
        if self.chk(self.a)==(self.p,self.q):return list(self.a),'primary'
        if parity and self.chk(self.b)==(self.p,self.q):return list(self.b),'repaired'
        return None,'lost'

class SemanticBank:
    def __init__(self,capacity,vocab,idf,comparative=True,mode='factor'):
        self.capacity=capacity;self.vocab=vocab;self.idf=idf;self.comparative=comparative;self.mode=mode;self.slots=[];self.writes=0;self.replacements=0;self.blocked=0;self.holds=0;self.margin=0.
    def retained_value(self,slot,step):
        age=max(0,step-slot.created_step)/max(1,step+1)
        return slot.U*math.sqrt(max(1e-9,math.log1p(slot.count)))/(1+.15*age)
    def candidate_value(self,row):
        return float(row.utility01*(.65+.35*row.stability)*(.75+.25*row.evidence_confidence))
    def admit(self,row,step,margin):
        if row.evidence_confidence <= 0 or row.utility01<.5:
            self.holds+=1;return False
        c=self.candidate_value(row)
        slot=Slot(row.text,self.vocab.encode(row.text),int(row.uid),float(row.utility01),float(row.stability),float(row.evidence_confidence),int(row['count']),int(row.first_pos),step)
        if len(self.slots)<self.capacity:
            self.slots.append(slot);self.writes+=1;return True
        vals=[self.retained_value(s,step) for s in self.slots];j=int(np.argmin(vals))
        if self.comparative:
            if c-vals[j] <= margin:self.blocked+=1;return False
        else:
            # Old/automatic full-bank behavior: qualifying candidate evicts weakest.
            if c<.5:self.blocked+=1;return False
        self.slots[j]=slot;self.writes+=1;self.replacements+=1;return True
    def corrupt(self,p,seed):
        rng=random.Random(seed)
        for s in self.slots:
            if rng.random()<p:s.corrupt(rng)
    def replay_ids(self,parity=True):
        out=[];repair=lost=0
        for s in self.slots:
            ids,status=s.retrieve(parity)
            if ids is not None:out.append(ids)
            repair+=status=='repaired';lost+=status=='lost'
        return out,repair,lost
    def query_score(self,q_tokens,slot):
        q=set(q_tokens);s=set(content_tokens(slot.text));ov=q&s
        if not ov:return 0.
        num=sum(self.idf.get(t,1.) for t in ov);den=sum(self.idf.get(t,1.) for t in q)+1e-12
        return float(num/den)
    def retrieve_templates(self,prompt,max_hops=2,parity=True):
        # Honest associative retrieval: never force a guess on tied/no-overlap addresses.
        q=content_tokens(prompt);chosen=[];seen=set();logs=[]
        for hop in range(max_hops):
            scores=[]
            for i,s in enumerate(self.slots):
                if i in seen:continue
                scores.append((self.query_score(q,s),i))
            if not scores:break
            scores.sort(reverse=True);best,bi=scores[0];second=scores[1][0] if len(scores)>1 else 0.
            # Dynamic margin: rare-token exact overlap should separate from the next address.
            margin=max(.025,.10*best)
            if best<=0 or best-second<=margin:
                logs.append({'hop':hop,'resolved':False,'best':best,'second':second,'margin':margin});break
            slot=self.slots[bi];ids,status=slot.retrieve(parity)
            if ids is None:
                logs.append({'hop':hop,'resolved':False,'best':best,'second':second,'margin':margin,'status':'lost'});break
            chosen.append(slot.text);seen.add(bi);q += content_tokens(slot.text)
            logs.append({'hop':hop,'resolved':True,'best':best,'second':second,'margin':margin,'status':status,'text':slot.text,'uid':slot.uid})
        return chosen,logs


class AdaptiveSemanticBank(SemanticBank):
    """Bounded soft-wall replacement: churn stiffens; persistent blocking softens."""
    def __init__(self,capacity,vocab,idf,comparative=True,adaptive=True,mode='v44'):
        super().__init__(capacity,vocab,idf,comparative,mode)
        self.adaptive=adaptive;self.base_margin=0.;self.current_margin=0.;self.margin_min=np.inf;self.margin_max=0.
        self.replace_ema=0.;self.block_ema=0.;self.pressure_alpha=.08
    def candidate_value(self,row):
        return float(row.utility01*(.65+.35*row.evidence_confidence)*(.75+.25*(1.-float(getattr(row,'interference_risk01',0.)))))
    def retained_value(self,slot,step):
        age=max(0,step-slot.created_step)/max(1,step+1)
        return float(slot.U*(.90+.10*slot.Q)/(1+.08*age))
    def _pressure(self,event):
        a=self.pressure_alpha
        self.replace_ema=(1-a)*self.replace_ema+a*(1. if event=='replace' else 0.)
        self.block_ema=(1-a)*self.block_ema+a*(1. if event=='block' else 0.)
        if self.adaptive:
            scale=float(np.clip(math.exp(1.25*(self.replace_ema-self.block_ema)),.25,2.5))
            self.current_margin=self.base_margin*scale
        self.margin_min=min(self.margin_min,self.current_margin);self.margin_max=max(self.margin_max,self.current_margin)
    def admit(self,row,step,margin=None):
        if row.evidence_confidence<=0 or row.utility01<.5:
            self.holds+=1;return False
        c=self.candidate_value(row)
        slot=Slot(row.text,self.vocab.encode(row.text),int(row.uid),float(row.utility01),float(row.stability),float(row.evidence_confidence),int(row['count']),int(row.first_pos),step)
        if len(self.slots)<self.capacity:
            self.slots.append(slot);self.writes+=1;return True
        vals=[self.retained_value(s,step) for s in self.slots];j=int(np.argmin(vals));m=self.current_margin if self.comparative else 0.
        # Novelty is pressure for admission, not utility. It may reduce the extra
        # replacement margin, but never allows a lower-value candidate to win.
        novelty=float(getattr(row,'novelty01',0.));m_eff=m*(1.-.45*novelty)
        if self.comparative and c-vals[j]<=m_eff:
            self.blocked+=1;self._pressure('block');return False
        self.slots[j]=slot;self.writes+=1;self.replacements+=1;self._pressure('replace');return True

def multi_gate_margin(cand):
    C=cand.utility01.to_numpy(float)*(.65+.35*cand.evidence_confidence.to_numpy(float))*(.75+.25*(1.-cand.interference_risk01.to_numpy(float)))
    q25,q75=np.quantile(C,[.25,.75]);return float(.18*max(1e-6,q75-q25))

def build_v44_bank(cand,vocab,kind='multi_adaptive_gate',capacity=96):
    idf=compute_idf(cand);comparative=('auto' not in kind);adaptive=('adaptive' in kind)
    bank=AdaptiveSemanticBank(capacity,vocab,idf,comparative=comparative,adaptive=adaptive,mode=kind)
    base=multi_gate_margin(cand);bank.base_margin=base;bank.current_margin=base;bank.margin=base;bank.margin_min=base;bank.margin_max=base
    for step,row in cand.sort_values('first_pos').iterrows():bank.admit(row,step,base)
    bank.margin=bank.current_margin
    return bank

class DualLaneBank(AdaptiveSemanticBank):
    """Shared-capacity bank that preserves emergent evidence lanes without semantic quotas."""
    def __init__(self,capacity,vocab,idf,mode='dual_lane_adaptive'):
        super().__init__(capacity,vocab,idf,comparative=True,adaptive=True,mode=mode)
        self.lane_pressure=Counter()
    def _lane(self,row):
        return str(getattr(row,'portfolio_lane','single'))
    def admit(self,row,step,margin=None):
        if row.evidence_confidence<=0 or row.utility01<.5:
            self.holds+=1;return False
        lane=self._lane(row);self.lane_pressure[lane]+=1
        c=self.candidate_value(row)
        slot=Slot(row.text,self.vocab.encode(row.text),int(row.uid),float(row.utility01),float(row.stability),float(row.evidence_confidence),int(row['count']),int(row.first_pos),step);slot.lane=lane
        if len(self.slots)<self.capacity:
            self.slots.append(slot);self.writes+=1;return True
        lanes=set(self.lane_pressure)|{getattr(x,'lane','single') for x in self.slots}
        total=max(1,sum(self.lane_pressure.values()))
        desired={l:max(1,int(round(self.capacity*self.lane_pressure[l]/total))) for l in lanes}
        # Repair rounding to capacity deterministically.
        while sum(desired.values())>self.capacity:
            l=max(sorted(desired),key=lambda x:desired[x]);desired[l]-=1
        while sum(desired.values())<self.capacity:
            l=max(sorted(desired),key=lambda x:self.lane_pressure[x]);desired[l]+=1
        counts=Counter(getattr(x,'lane','single') for x in self.slots)
        if counts[lane] < desired.get(lane,0):
            over=[l for l in lanes if counts[l]>desired.get(l,0)]
            pool=[i for i,x in enumerate(self.slots) if getattr(x,'lane','single') in over] if over else list(range(len(self.slots)))
        else:
            pool=[i for i,x in enumerate(self.slots) if getattr(x,'lane','single')==lane]
            if not pool:pool=list(range(len(self.slots)))
        vals=[self.retained_value(self.slots[i],step) for i in pool];pi=int(np.argmin(vals));j=pool[pi];weak=vals[pi]
        m=self.current_margin;novelty=float(getattr(row,'novelty01',0.));m_eff=m*(1.-.45*novelty)
        if c-weak<=m_eff:
            self.blocked+=1;self._pressure('block');return False
        self.slots[j]=slot;self.writes+=1;self.replacements+=1;self._pressure('replace');return True

def build_dual_lane_bank(cand,vocab,capacity=96):
    idf=compute_idf(cand);bank=DualLaneBank(capacity,vocab,idf)
    base=multi_gate_margin(cand);bank.base_margin=base;bank.current_margin=base;bank.margin=base;bank.margin_min=base;bank.margin_max=base
    for step,row in cand.sort_values('first_pos').iterrows():bank.admit(row,step,base)
    bank.margin=bank.current_margin
    return bank

class SplitRoleBank:
    """One total memory budget, two learned operational roles.

    Replay/consolidation and query-time addressing compete under separate half-budget
    banks because the local ablation showed they optimize different consequences.
    This 50/50 split is an explicit scaffold and is logged for later adaptive removal.
    """
    def __init__(self,replay_bank,query_bank,total_capacity):
        self.replay_bank=replay_bank;self.query_bank=query_bank;self.capacity=total_capacity;self.mode='split_role'
    @property
    def slots(self):return self.query_bank.slots
    @property
    def writes(self):return self.replay_bank.writes+self.query_bank.writes
    @property
    def replacements(self):return self.replay_bank.replacements+self.query_bank.replacements
    @property
    def blocked(self):return self.replay_bank.blocked+self.query_bank.blocked
    @property
    def holds(self):return self.replay_bank.holds+self.query_bank.holds
    @property
    def margin(self):return (getattr(self.replay_bank,'margin',0.)+getattr(self.query_bank,'margin',0.))/2
    @property
    def margin_min(self):return min(getattr(self.replay_bank,'margin_min',self.margin),getattr(self.query_bank,'margin_min',self.margin))
    @property
    def margin_max(self):return max(getattr(self.replay_bank,'margin_max',self.margin),getattr(self.query_bank,'margin_max',self.margin))
    @property
    def replace_ema(self):return getattr(self.replay_bank,'replace_ema',0.)+getattr(self.query_bank,'replace_ema',0.)
    @property
    def block_ema(self):return getattr(self.replay_bank,'block_ema',0.)+getattr(self.query_bank,'block_ema',0.)
    def corrupt(self,p,seed):
        self.replay_bank.corrupt(p,seed);self.query_bank.corrupt(p,seed+100003)
    def replay_ids(self,parity=True):
        a,ra,la=self.replay_bank.replay_ids(parity);_,rb,lb=self.query_bank.replay_ids(parity);return a,ra+rb,la+lb
    def retrieve_templates(self,prompt,max_hops=2,parity=True):
        return self.query_bank.retrieve_templates(prompt,max_hops,parity)

def build_split_role_bank(cand_v43,cand_multi,vocab,capacity=96):
    replay_cap=capacity//2;query_cap=capacity-replay_cap
    replay=build_v44_bank(cand_multi,vocab,'multi_auto',replay_cap)
    query=build_bank(cand_v43,{},vocab,'factor_gate',query_cap)
    return SplitRoleBank(replay,query,capacity)

def compute_idf(cand):
    docs=[set(content_tokens(s)) for s in cand.text];df=Counter();
    for d in docs:df.update(d)
    N=len(docs);return {t:(math.log((N+1)/(c+1))+1)**2 for t,c in df.items()}

def add_stability(cand):
    x=cand.copy();span=x.temporal_span.to_numpy(float);cnt=np.log1p(x['count'].to_numpy(float));cnt=cnt/(cnt.max()+1e-12);x['stability']=np.clip(.5*span+.5*cnt,0,1);return x

def gate_margin(cand):
    # Field report says the margin is calibration-dependent, not universal.
    # Here it scales to candidate-value dispersion using no semantic labels.
    C=cand.utility01.to_numpy(float)*(.65+.35*cand.stability.to_numpy(float))*(.75+.25*cand.evidence_confidence.to_numpy(float))
    return float(.25*np.std(C))

def build_bank(cand,meta,vocab,kind='factor_gate',capacity=96):
    idf=compute_idf(cand);bank=SemanticBank(capacity,vocab,idf,comparative=('auto' not in kind),mode=kind)
    margin=gate_margin(cand);bank.margin=margin
    # Process in observational order, not best-first.
    for step,row in cand.sort_values('first_pos').iterrows():bank.admit(row,step,margin)
    return bank

def old_v42_bank(train,vocab,final_model,capacity=96):
    # v4.2 selector, but bank is unique-template bounded for fair address retrieval.
    pos=defaultdict(list)
    for i,s in enumerate(train):pos[s].append(i)
    docs=list(pos.keys());counts=Counter();toks=[]
    for s in docs:
        q=content_tokens(s);toks.append(q);counts.update(set(q))
    rows=[]
    for i,(s,q) in enumerate(zip(docs,toks)):
        rows.append({'uid':i,'text':s,'first_pos':min(pos[s]),'count':len(pos[s]),
                     'recurrence':np.mean([math.log1p(counts[w]) for w in q]) if q else 0.,
                     'cross_context':np.mean([counts[w]>=3 for w in q]) if q else 0.,
                     'predictability':-seq_loss(final_model,vocab.encode(s))})
    d=pd.DataFrame(rows);X=d[['recurrence','cross_context','predictability']].to_numpy(float);Z,_,_,active=zcols(X)
    C=np.zeros((3,3));idx=np.where(active)[0]
    if len(idx)>1:
        Q=np.nan_to_num(np.abs(np.corrcoef(Z[:,idx],rowvar=False)));np.fill_diagonal(Q,0)
        for a,i in enumerate(idx):
            for b,j in enumerate(idx):C[i,j]=Q[a,b]
    rho=np.where(active,.7,0);a=rho**3;base=a/(a.sum()+1e-12);w=a/(1+1.5*(C@base));w/=w.sum()+1e-12;score=Z@w
    sets=[set(content_tokens(s)) for s in docs];selected=[];remain=set(range(len(d)))
    while remain and len(selected)<min(capacity,len(d)):
        best=None;bv=-1e99
        for i in remain:
            red=max((len(sets[i]&sets[j])/(len(sets[i]|sets[j])+1e-12) for j in selected),default=0.);v=float(score[i])-.85*red
            if v>bv:bv=v;best=i
        selected.append(best);remain.remove(best)
    # Reuse bank container, direct add selected templates.
    idf=compute_idf(pd.DataFrame({'text':docs}));bank=SemanticBank(capacity,vocab,idf,comparative=False,mode='v42_selector')
    for step,i in enumerate(selected):
        row=d.iloc[i];slot=Slot(row.text,vocab.encode(row.text),int(row.uid),.75,.75,.75,int(row['count']),int(row.first_pos),step);bank.slots.append(slot);bank.writes+=1
    return bank

def random_bank(train,vocab,capacity,seed):
    docs=list(dict.fromkeys(train));rng=random.Random(seed);picks=rng.sample(docs,min(capacity,len(docs)));tmp=pd.DataFrame({'text':docs});idf=compute_idf(tmp);bank=SemanticBank(capacity,vocab,idf,comparative=False,mode='random')
    pos={s:i for i,s in enumerate(docs)}
    for step,s in enumerate(picks):bank.slots.append(Slot(s,vocab.encode(s),pos[s],.5,.5,.5,1,pos[s],step));bank.writes+=1
    return bank

# -----------------------------------------------------------------------------
# Binding-specific probes. No chain ids/answers enter training or memory selection.
# -----------------------------------------------------------------------------

def augment_prompt(bank,prompt,parity=True,order='path'):
    if bank is None:return prompt,[],[]
    templates,logs=bank.retrieve_templates(prompt,2,parity)
    if order=='reverse':templates=list(reversed(templates))
    aug=(' '.join(templates)+' '+prompt).strip() if templates else prompt
    return aug,templates,logs

def probe_panel(model,vocab,chains,seed,stage,variant,bank=None,parity=True):
    rows=[]
    def add(kind,prompt,ans,item,mode='plain',candidates=None,wrong=None):
        aug=prompt;retr=[];logs=[]
        if mode in ('assist','assist_reverse') and bank is not None:
            aug,retr,logs=augment_prompt(bank,prompt,parity,'reverse' if mode=='assist_reverse' else 'path')
        elif mode in ('first_only','second_only') and bank is not None:
            # Single-premise controls separate retrieval from composition.
            _,rr,ll=augment_prompt(bank,prompt,parity,'path');retr=(rr[:1] if mode=='first_only' else rr[-1:]) if rr else [];logs=ll;aug=(' '.join(retr)+' '+prompt).strip() if retr else prompt
        r=token_rank(model,vocab,aug,ans);cr=np.nan;margin=np.nan;wrong_score=np.nan;true_score=np.nan
        if candidates:
            cr,sc=candidate_rank(model,vocab,aug,ans,candidates);true_score=sc.get(ans,np.nan)
            if wrong is not None:wrong_score=sc.get(wrong,np.nan);margin=true_score-wrong_score
        rows.append({'seed':seed,'stage':stage,'variant':variant,'probe':kind,'mode':mode,'item':item,'prompt':prompt,'aug_prompt':aug,'answer':ans,
                     'rank':r,'candidate_rank':cr,'binding_margin':margin,'true_score':true_score,'wrong_score':wrong_score,
                     'top1':int(r==1),'top5':int(r<=5),'candidate_top1':int(cr==1) if not np.isnan(cr) else np.nan,
                     'top_tokens':'|'.join(top_tokens(model,vocab,aug,5)),'retrieved':' || '.join(retr),'retrieval_log':json.dumps(logs)})
    for i,(p,a) in enumerate(FACTS):add('exact_fact',p,a,f'fact{i}')
    for i,(p,a) in enumerate(REPHRASE):add('rephrase',p,a,f'rephrase{i}')
    q=chains[chains.kind=='probe'].reset_index(drop=True);zcands=q.z.tolist();xcands=q.x.tolist()
    for i,row in q.iterrows():
        wrong=q.iloc[(i+1)%len(q)]
        for mode in ['plain','assist','assist_reverse','first_only','second_only']:
            add('two_hop',f'{row.x} ultimately maps to',row.z,f'chain{i}',mode,zcands,wrong.z)
        add('two_hop_shuffled',f'{row.x} ultimately maps to',wrong.z,f'chain{i}','plain',zcands,row.z)
        for mode in ['plain','assist']:
            add('inverse',f'{row.y} is contained by',row.x,f'chain{i}',mode,xcands,wrong.x)
        add('premise_contains',f'{row.x} contains',row.y,f'chain{i}')
        add('premise_maps',f'{row.y} maps to',row.z,f'chain{i}')
    return pd.DataFrame(rows)

def summarize_probes(df):
    def safe_cmrr(x):
        a=np.asarray(x,float);a=a[np.isfinite(a)]
        return float(np.mean(1/a)) if len(a) else np.nan
    return df.groupby(['stage','variant','probe','mode']).agg(top1=('top1','mean'),top5=('top5','mean'),candidate_top1=('candidate_top1','mean'),
        mrr=('rank',lambda x:float(np.mean(1/np.asarray(x,float)))),candidate_mrr=('candidate_rank',safe_cmrr),
        binding_margin=('binding_margin','mean'),median_rank=('rank','median'),n=('rank','size')).reset_index()

VARIANTS=['baseline','random_bank','v42_selector','v43_factor_gate','v44_multi_auto','v44_multi_fixed_gate','v44_multi_adaptive_gate','v44_dual_lane_adaptive']

def run_one(seed=0,full=False,corpus='interleaved',device=None,capacity=96,contextual_demo=False):
    seed_all(seed);train,chains=make_training_corpus(seed,contextual_demo=contextual_demo);stream=natural_stream(seed,corpus,700) if full else proxy_stream(seed,180);vocab=Vocab(train+stream)
    if device is None:device='cuda' if full and torch.cuda.is_available() else 'cpu'

    # Shared initial learner: all variants branch from identical weights.
    base=TinySSM(len(vocab.itos)).to(device);opt=torch.optim.Adam(base.parameters(),lr=.006);order=list(range(len(train)));early_state=None;mid_state=None
    for ep in range(3):
        random.Random(seed+ep).shuffle(order)
        for i in order:train_sentence(base,opt,vocab.encode(train[i]))
        if ep==0:early_state=copy.deepcopy(base.state_dict())
        if ep==1:mid_state=copy.deepcopy(base.state_dict())
    final_state=copy.deepcopy(base.state_dict())
    early=TinySSM(len(vocab.itos)).to(device);early.load_state_dict(early_state)
    mid=TinySSM(len(vocab.itos)).to(device);mid.load_state_dict(mid_state)
    cand=unique_candidate_table(train,vocab,early,mid,base);cand=add_stability(cand)
    cand_v43,meta_v43=evidence_model(cand,'factor')
    cand_multi,meta_multi=multi_consequence_model(cand,'factor');cand_portfolio=portfolio_model(cand_v43,cand_multi)

    # Banks are built without semantic/fact/chain labels.
    banks={
      'baseline':None,
      'random_bank':random_bank(train,vocab,capacity,seed+999),
      'v42_selector':old_v42_bank(train,vocab,base,capacity),
      'v43_factor_gate':build_bank(cand_v43,meta_v43,vocab,'factor_gate',capacity),
      'v44_multi_auto':build_v44_bank(cand_multi,vocab,'multi_auto',capacity),
      'v44_multi_fixed_gate':build_v44_bank(cand_multi,vocab,'multi_fixed_gate',capacity),
      'v44_multi_adaptive_gate':build_v44_bank(cand_multi,vocab,'multi_adaptive_gate',capacity),
      'v44_dual_lane_adaptive':build_dual_lane_bank(cand_portfolio,vocab,capacity),
    }
    runs=[];probes=[];bank_audit=[]
    qprobe=chains[chains.kind=='probe'].reset_index(drop=True)
    premise_text=set()
    for _,r in qprobe.iterrows():premise_text|={f'{r.x} contains {r.y}.',f'{r.y} maps to {r.z}.'}

    for variant in VARIANTS:
        model=TinySSM(len(vocab.itos)).to(device);model.load_state_dict(final_state);opt=torch.optim.Adam(model.parameters(),lr=.006);bank=banks[variant]
        # Evaluate pre-interference plain behavior only once per variant.
        probes.append(probe_panel(model,vocab,chains,seed,'pre',variant,None,True))
        repair=lost=0;replay=[]
        if bank is not None:
            bank.corrupt(.18,seed+1234);replay,repair,lost=bank.replay_ids(True)
        total=0
        for ep in range(2 if full else 1):
            for s in stream:
                train_sentence(model,opt,vocab.encode(s));total+=1
                if bank is not None and replay and total%6==0:train_sentence(model,opt,replay[(total//6)%len(replay)])
        post=probe_panel(model,vocab,chains,seed,'post',variant,bank,True);probes.append(post)

        def mrr(kind,mode='plain'):
            q=post[(post.probe==kind)&(post['mode']==mode)];return float(np.mean(1/q['rank'].to_numpy(float))) if len(q) else np.nan
        def cmrr(kind,mode):
            q=post[(post.probe==kind)&(post['mode']==mode)];x=q.candidate_rank.to_numpy(float);return float(np.nanmean(1/x)) if len(q) else np.nan
        def ctop(kind,mode):
            q=post[(post.probe==kind)&(post['mode']==mode)];return float(np.nanmean(q.candidate_top1)) if len(q) else np.nan
        def bmargin(kind,mode):
            q=post[(post.probe==kind)&(post['mode']==mode)];return float(np.nanmean(q.binding_margin)) if len(q) else np.nan
        # direct true-vs-shuffled margin plus transparent address-path audit.
        plain=post[(post.probe=='two_hop')&(post['mode']=='plain')]
        assist=post[(post.probe=='two_hop')&(post['mode']=='assist')]
        rev=post[(post.probe=='two_hop')&(post['mode']=='assist_reverse')]
        first=post[(post.probe=='two_hop')&(post['mode']=='first_only')];second=post[(post.probe=='two_hop')&(post['mode']=='second_only')]
        target_hits=[];two_template=[]
        for _,ar in assist.iterrows():
            rt='' if pd.isna(ar.retrieved) else str(ar.retrieved)
            target_hits.append(str(ar.answer).lower() in set(content_tokens(rt)))
            two_template.append('||' in rt)
        if bank is not None:
            stored={s.text for s in bank.slots};premise_cov=len(stored & premise_text)/max(1,len(premise_text))
            both=0
            for _,r in qprobe.iterrows():both+=int(f'{r.x} contains {r.y}.' in stored and f'{r.y} maps to {r.z}.' in stored)
            both/=len(qprobe)
            ba={'seed':seed,'variant':variant,'capacity':capacity,'slots':len(bank.slots),'writes':bank.writes,'replacements':bank.replacements,'blocked':bank.blocked,'holds':bank.holds,'margin':bank.margin,'margin_min':getattr(bank,'margin_min',bank.margin),'margin_max':getattr(bank,'margin_max',bank.margin),'replace_ema':getattr(bank,'replace_ema',0.),'block_ema':getattr(bank,'block_ema',0.),'premise_template_coverage':premise_cov,'both_premises_stored':both,'repair':repair,'lost':lost}
            bank_audit.append(ba)
        runs.append({'seed':seed,'variant':variant,'full':full,'corpus':corpus,
          'exact_mrr':mrr('exact_fact'),'rephrase_mrr':mrr('rephrase'),'premise_contains_mrr':mrr('premise_contains'),'premise_maps_mrr':mrr('premise_maps'),
          'twohop_plain_mrr':mrr('two_hop','plain'),'twohop_assist_mrr':mrr('two_hop','assist'),'twohop_reverse_mrr':mrr('two_hop','assist_reverse'),'twohop_first_only_mrr':mrr('two_hop','first_only'),'twohop_second_only_mrr':mrr('two_hop','second_only'),
          'twohop_plain_candidate_mrr':cmrr('two_hop','plain'),'twohop_assist_candidate_mrr':cmrr('two_hop','assist'),'twohop_reverse_candidate_mrr':cmrr('two_hop','assist_reverse'),'twohop_first_only_candidate_mrr':cmrr('two_hop','first_only'),'twohop_second_only_candidate_mrr':cmrr('two_hop','second_only'),
          'twohop_plain_candidate_top1':ctop('two_hop','plain'),'twohop_assist_candidate_top1':ctop('two_hop','assist'),'twohop_reverse_candidate_top1':ctop('two_hop','assist_reverse'),'twohop_first_only_candidate_top1':ctop('two_hop','first_only'),'twohop_second_only_candidate_top1':ctop('two_hop','second_only'),
          'binding_margin_plain':bmargin('two_hop','plain'),'binding_margin_assist':bmargin('two_hop','assist'),'binding_margin_reverse':bmargin('two_hop','assist_reverse'),'binding_margin_first_only':bmargin('two_hop','first_only'),'binding_margin_second_only':bmargin('two_hop','second_only'),
          'composition_gain_candidate_mrr':cmrr('two_hop','assist')-max(cmrr('two_hop','first_only'),cmrr('two_hop','second_only')),
          'composition_gain_binding_margin':bmargin('two_hop','assist')-max(bmargin('two_hop','first_only'),bmargin('two_hop','second_only')),
          'retrieval_target_hit_rate':float(np.mean(target_hits)) if target_hits else 0.,'retrieval_two_template_rate':float(np.mean(two_template)) if two_template else 0.,
          'inverse_plain_mrr':mrr('inverse','plain'),'inverse_assist_mrr':mrr('inverse','assist'),'parity_repaired':repair,'parity_lost':lost,
          'stream_loss':float(np.mean([seq_loss(model,vocab.encode(s)) for s in stream[-60:]]))})

    evidence_audit={'seed':seed,'v43_single_anchor':meta_v43,'v44_multi_consequence':meta_multi,'portfolio_lane_counts':cand_portfolio.portfolio_lane.value_counts().to_dict()}
    return pd.DataFrame(runs),pd.concat(probes,ignore_index=True),pd.DataFrame(bank_audit),cand_portfolio,evidence_audit

def main(argv=None):
    ap=argparse.ArgumentParser();ap.add_argument('--full',action='store_true');ap.add_argument('--corpus',choices=['shakespeare','bible','interleaved'],default='interleaved');ap.add_argument('--seeds',type=int,default=3);ap.add_argument('--capacity',type=int,default=96);ap.add_argument('--contextual-demo',action='store_true',help='Add contiguous demonstration-chain contexts; probe targets remain omitted.');args=ap.parse_args(argv)
    rr=[];pp=[];bb=[];cc=[];ee=[]
    for seed in range(args.seeds):
        print('seed',seed);r,p,b,c,e=run_one(seed,args.full,args.corpus,capacity=args.capacity,contextual_demo=args.contextual_demo);rr.append(r);pp.append(p);bb.append(b);c['seed']=seed;cc.append(c);ee.append(e)
        print(r[['variant','exact_mrr','twohop_assist_candidate_mrr','composition_gain_candidate_mrr','binding_margin_assist']].to_string(index=False))
    runs=pd.concat(rr,ignore_index=True);probes=pd.concat(pp,ignore_index=True);banks=pd.concat(bb,ignore_index=True);cand=pd.concat(cc,ignore_index=True);tag=(('full_'+args.corpus) if args.full else 'proxy')+('_contextual' if args.contextual_demo else '_legacy')
    runs.to_csv(OUT/f'{tag}_{args.seeds}seed_runs.csv',index=False);probes.to_csv(OUT/f'{tag}_{args.seeds}seed_probe_details.csv',index=False);banks.to_csv(OUT/f'{tag}_{args.seeds}seed_bank_audit.csv',index=False);cand.to_csv(OUT/f'{tag}_{args.seeds}seed_candidate_evidence.csv',index=False);(OUT/f'{tag}_{args.seeds}seed_evidence_models.json').write_text(json.dumps(ee,indent=2));summarize_probes(probes).to_csv(OUT/f'{tag}_{args.seeds}seed_probe_summary.csv',index=False)
    med=runs.groupby('variant').median(numeric_only=True)
    def vals(v,col): return runs[runs.variant==v].sort_values('seed')[col].to_numpy(float)
    checks={
      'multi_auto_exact_beats_v43_seed_fraction':float(np.mean(vals('v44_multi_auto','exact_mrr')>vals('v43_factor_gate','exact_mrr'))),
      'multi_auto_assisted_binding_margin_positive_seed_fraction':float(np.mean(vals('v44_multi_auto','binding_margin_assist')>0)),
      'multi_auto_assisted_mrr_beats_v43_seed_fraction':float(np.mean(vals('v44_multi_auto','twohop_assist_candidate_mrr')>vals('v43_factor_gate','twohop_assist_candidate_mrr'))),
      'adaptive_gate_assisted_mrr_beats_auto_seed_fraction':float(np.mean(vals('v44_multi_adaptive_gate','twohop_assist_candidate_mrr')>vals('v44_multi_auto','twohop_assist_candidate_mrr'))),
      'dual_lane_retrieval_target_hit_rate_median':float(np.median(vals('v44_dual_lane_adaptive','retrieval_target_hit_rate'))),
      'dual_lane_binding_margin_positive_seed_fraction':float(np.mean(vals('v44_dual_lane_adaptive','binding_margin_assist')>0)),
      'dual_lane_strict_composition_gain_positive_seed_fraction':float(np.mean(vals('v44_dual_lane_adaptive','composition_gain_candidate_mrr')>0)),
      'parity_repairs_faults':bool(np.median(vals('v44_multi_adaptive_gate','parity_repaired'))>0),
    }
    (OUT/f'{tag}_{args.seeds}seed_checks.json').write_text(json.dumps(checks,indent=2))
    bankmed=banks.groupby('variant').median(numeric_only=True) if len(banks) else pd.DataFrame()
    report=f'''Adrianic v4.4 — Multi-Consequence Value + Adaptive Binding Gate

Mode: {tag}
Seeds: {args.seeds}
Capacity: {args.capacity}
Contextual demonstration ingestion: {args.contextual_demo}

What changed from v4.3
======================
1. Permanent-memory value is no longer anchored to one future-predictability scalar.
2. Four positive consequence channels are dependency-discounted: two learning-window gains, future support, and contextual retrieval usefulness. Novelty and interference remain separate structural pressures rather than being collapsed into positive value.
3. The latent-factor/common-cause machinery operates over those measured consequences; correlated reasons cannot vote independently.
4. A dual-lane ablation preserves predictive and direct-consequence support without collapsing them immediately to one scalar.
5. The comparative replacement wall is adaptive: destructive churn stiffens it, persistent blocking softens it, within hard bounds.
6. Honest IDF addressing, raw-template parity protection and unresolved retrieval are retained.
7. Retrieval success and reasoning success are separated. first_only and second_only controls are scored; strict composition gain is full-context performance minus the better single-premise control.

Controls
========
- baseline
- random_bank
- v42_selector
- v43_factor_gate: previous single-anchor factor + fixed comparative gate
- v44_multi_auto: multi-consequence value + automatic replacement
- v44_multi_fixed_gate: multi-consequence value + fixed comparative replacement
- v44_multi_adaptive_gate: multi-consequence value + adaptive replacement
- v44_dual_lane_adaptive: shared-capacity, pressure-balanced predictive/direct-consequence lanes + adaptive replacement

The composed X->Z target remains absent for all probe entities.

Median run metrics
==================
{med[['exact_mrr','rephrase_mrr','premise_contains_mrr','premise_maps_mrr','twohop_plain_candidate_mrr','twohop_assist_candidate_mrr','twohop_first_only_candidate_mrr','twohop_second_only_candidate_mrr','twohop_reverse_candidate_mrr','binding_margin_assist','binding_margin_first_only','binding_margin_second_only','composition_gain_candidate_mrr','composition_gain_binding_margin','retrieval_target_hit_rate','inverse_plain_mrr','inverse_assist_mrr','stream_loss','parity_repaired','parity_lost']].to_string()}

Median bank audit
=================
{bankmed.to_string() if len(bankmed) else 'none'}

Checks
======
{json.dumps(checks,indent=2)}

Interpretation discipline
=========================
No v4.4 variant is declared a winner a priori. Retrieval, replay retention, churn and strict composition are reported separately. Retrieving the correct Y->Z premise is useful binding/address behavior, but it is not itself proof of two-hop composition. The strict composition score requires the full two-premise context to beat BOTH single-premise controls. If that remains <=0, the honest conclusion is that memory binding improved while compositional reasoning remains unresolved.

All value channels, novelty/interference pressures, and learned dependency/factor weights are logged for ablation. No semantic/fact/chain labels enter memory valuation or replacement.
'''
    (OUT/f'{tag}_{args.seeds}seed_REPORT.txt').write_text(report);print(report)



## Full natural-corpus run

Balanced Tiny Shakespeare + public-domain KJV interference:

```python
main(["--full", "--corpus", "interleaved", "--seeds", "3", "--contextual-demo"])
```


In [ ]:
main(["--full", "--corpus", "interleaved", "--seeds", "3", "--contextual-demo"])

## Optional larger run

If the 3-seed mechanics are clean:

```python
main(["--full", "--corpus", "interleaved", "--seeds", "8", "--contextual-demo"])
```

The strict composition metric must beat both single-premise controls; raw retrieval alone is not counted as reasoning.

## Verified local eight-seed result

# Adrianic v4.4 — Multi-Consequence Value + Adaptive Binding Gate

## Eight-seed local proxy verification

This version was built to address the two main v4.3 failures:

1. **single-anchor value collapse** — v4.3 let one self-supervised future-predictability signal dominate memory value;
2. **fixed replacement rigidity** — the comparative gate reduced churn but could block useful plasticity.

v4.4 keeps the existing parity-protected raw templates, honest associative addressing,
`VALUE_UNRESOLVED` behavior, and direct binding controls.

## Structural changes

### Multi-consequence value

Permanent-memory value is no longer defined by one future-predictability scalar.

Four *positive consequence* channels are measured and dependency-discounted:

- prediction improvement from epoch 1 -> epoch 2,
- prediction improvement from epoch 2 -> epoch 3,
- persistence/support across later chronological observation windows,
- contextual usefulness on later related observations.

The latent common-cause machinery operates on these measured consequence channels.

**Novelty and interference are deliberately kept separate from positive value.**

- novelty may soften the extra replacement margin,
- unrelated-context interference acts as a cost,
- neither can independently turn a low-value candidate into a high-value memory.

That separation was introduced after an initial v4.4 screen showed that treating novelty and
"low interference" as ordinary positive votes caused them to dominate simply because they were
less correlated with the other channels.

### Adaptive comparative replacement

The replacement wall is now a bounded soft wall.

- repeated replacement pressure stiffens it,
- repeated blocked-candidate pressure softens it,
- the wall is never treated as a universal target.

### Parallel-lane ablation

A dual-lane variant keeps the older predictive structural estimator and the new direct-consequence
estimator as separate support lanes under one shared memory budget. This tests whether premature
scalarization is itself destructive.

### Retrieval vs reasoning

The probe now includes:

- full retrieved context,
- first-premise-only,
- second-premise-only,
- reverse-order context,
- matched wrong target.

The strict composition score is:

`full-context candidate MRR - max(first-only MRR, second-only MRR)`.

A positive retrieval result is therefore not silently counted as reasoning.

## Eight-seed results

Median metrics:

```text
                         exact_mrr  rephrase_mrr  twohop_assist_candidate_mrr  binding_margin_assist  composition_gain_candidate_mrr  retrieval_target_hit_rate  stream_loss
variant                                                                                                                                                                     
baseline                  0.387067      0.047078                     0.304425              -0.068553                        0.000000                       0.00     0.883345
random_bank               0.518206      0.042409                     0.293036              -0.117969                        0.000000                       0.05     0.877689
v42_selector              0.486730      0.122365                     0.270397               0.048067                        0.000000                       0.00     0.885775
v43_factor_gate           0.369026      0.037470                     0.266389               0.021746                       -0.014385                       0.80     0.878367
v44_dual_lane_adaptive    0.421771      0.040477                     0.274980              -0.038704                       -0.027282                       0.45     0.871695
v44_multi_adaptive_gate   0.409911      0.049533                     0.302639               0.065262                        0.000000                       0.00     0.875830
v44_multi_auto            0.437326      0.084960                     0.350456               0.083197                        0.000000                       0.00     0.876543
v44_multi_fixed_gate      0.423703      0.049533                     0.296667               0.065262                        0.000000                       0.00     0.882975
```

Median bank behavior:

```text
                         writes  replacements  blocked  holds  premise_template_coverage  both_premises_stored  repair  lost
variant                                                                                                                     
random_bank                96.0           0.0      0.0    0.0                      0.300                  0.05    16.5   0.0
v42_selector               96.0           0.0      0.0    0.0                      0.000                  0.00    17.5   0.0
v43_factor_gate            98.5           2.5     28.5  146.5                      0.875                  0.80    18.5   0.0
v44_dual_lane_adaptive    105.5           9.5    100.0   66.5                      0.600                  0.45    18.0   0.0
v44_multi_adaptive_gate   108.5          12.5     32.5  130.5                      0.125                  0.00    16.5   0.0
v44_multi_auto            141.5          45.5      0.0  130.5                      0.075                  0.00    17.5   0.0
v44_multi_fixed_gate      108.0          12.0     33.5  130.5                      0.125                  0.00    17.0   0.0
```

## What actually improved

The strongest v4.4 **replay/consolidation** variant locally was `v44_multi_auto`.

Compared with the older v4.3 factor gate, across eight seeds:

- exact factual MRR was higher in **75%** of seeds;
- assisted candidate MRR was higher in **88%** of seeds;
- assisted true-vs-wrong binding margin was positive in **62%** of seeds.

Its median metrics were:

- exact MRR: **0.437** vs v4.3 **0.369**
- rephrase MRR: **0.085** vs v4.3 **0.037**
- assisted candidate MRR: **0.350** vs v4.3 **0.266**
- binding margin: **0.083** vs v4.3 **0.022**

However, its probe-time target retrieval was essentially absent. Its improvement comes from
**what replay does to the learner during interference**, not from successful two-hop associative
recall at question time.

## What the adaptive gate did

The adaptive gate strongly reduced churn:

- multi-auto median replacements: **45.5**
- multi-adaptive median replacements: **12.5**

But the adaptive gate beat automatic replacement on assisted candidate MRR in only
**25%** of seeds.

So the stability/plasticity tradeoff remains real. The soft wall is wired in and functioning,
but this calibration is still too conservative to declare it superior.

## What remains unsolved

The old v4.3 factor gate still had the best query-time relation retrieval:

- v4.3 median target-hit rate: **80%**
- dual-lane v4.4 median target-hit rate: **45%**

The dual-lane binding margin was positive in **50%**
of seeds, while strict composition gain was positive in only
**25%**.

Therefore:

> v4.4 improves the consolidation/replay value problem, but it does **not** establish two-hop
> compositional reasoning and it does not yet unify the best replay policy with the best
> query-time retrieval policy.

A naive fixed 50/50 split-role screen was also tried locally and did not improve the result, so
it was not promoted into the main harness.

## Natural-corpus next test

Run the supplied Colab notebook:

```python
main(["--full", "--corpus", "interleaved", "--seeds", "3", "--contextual-demo"])
```

This is the full Shakespeare/KJV interference test.

The key questions are now narrower:

1. Does multi-consequence replay reproduce its local advantage over the v4.3 single-anchor controller?
2. How much performance is lost when the adaptive gate removes churn?
3. Does the dual-lane variant preserve useful query-time binding under natural interference?
4. Does **any** variant produce positive copy-debiased composition gain?

No positive answer is assumed.
